<a href="https://colab.research.google.com/github/saivenkat-954/Multi-Agent-AI-System-Design/blob/main/Google_GenAI_APAC_Hackathon_Prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import auth
auth.authenticate_user()

# --- EDIT THESE ---
PROJECT_ID = "your-gcp-project-id"
REGION = "us-central1"
SERVICE_NAME = "agent-orchestrator"
# ------------------

!gcloud config set project {PROJECT_ID}
# Enable Cloud Run and Cloud Build APIs
!gcloud services enable run.googleapis.com cloudbuild.googleapis.com

In [ ]:
import os
from fastapi import FastAPI
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

app = FastAPI()

# Specialist Tool 1: Database/Task Agent Logic
@tool
def task_db_tool(query: str):
    """Adds or retrieves tasks from the persistent task database."""
    return f"DB_SUCCESS: Action recorded for '{query}'"

# Specialist Tool 2: Calendar Agent Logic
@tool
def calendar_tool(event_details: str):
    """Schedules or checks events on the user calendar."""
    return f"CALENDAR_SUCCESS: Event '{event_details}' scheduled."

# Primary Orchestrator Setup
llm = ChatOpenAI(model="gpt-4-turbo")
tools = [task_db_tool, calendar_tool]
agent_executor = create_react_agent(llm, tools)

class AgentRequest(BaseModel):
    prompt: str

@app.post("/execute")
async def run_agent(request: AgentRequest):
    inputs = {"messages": [("user", request.prompt)]}
    result = await agent_executor.ainvoke(inputs)
    return {"answer": result["messages"][-1].content}

@app.get("/health")
def health():
    return {"status": "ok"}

In [ ]:
# Deployment Command
!gcloud run deploy {SERVICE_NAME} \
    --source . \
    --region {REGION} \
    --allow-unauthenticated \
    --set-env-vars OPENAI_API_KEY="your-openai-key"

In [ ]:
import gradio as gr
import requests

# PASTE YOUR CLOUD RUN URL HERE (e.g., https://agent-xxx.a.run.app)
DEPLOYED_URL = "https://your-service-url-here.a.run.app"

def call_deployed_agent(message, history):
    try:
        response = requests.post(
            f"{DEPLOYED_URL}/execute",
            json={"prompt": message},
            timeout=30
        )
        return response.json()["answer"]
    except Exception as e:
        return f"Error connecting to Cloud Run: {str(e)}"

# Launch the UI inside Colab
demo = gr.ChatInterface(
    fn=call_deployed_agent,
    title="Multi-Agent System (Live on Cloud Run)",
    description="I am running on Google Cloud! Ask me to manage your tasks and calendar."
)
demo.launch()